# Solution 02 — the model shoot-out

Approach: the same prompt goes through every model you can reach, and `result.usage` goes into one row per
model. A `cost` of `None` means pydantic-ai doesn't know that model's price, so those rows are left out of the
"cheapest" pick. Otherwise an unpriced model would look free.

Notes:

- Token counts (and prices) differ per model even for the same prompt, so "which model" is a decision you can
  back with data.
- Add any model your keys allow to `candidates`.

In [ ]:
from learningdantic import setup

MODEL = setup()

In [ ]:
import os

from pydantic_ai import Agent

prompt = 'Explain what a "token" is, in exactly one sentence.'
candidates = {"MODEL": MODEL, "gpt-6.1-sol": "openai:gpt-6.1-sol"}
if os.getenv("ANTHROPIC_API_KEY"):
    candidates["claude-haiku-5-5"] = "anthropic:claude-haiku-5-5"

rows = []
for label, model_string in candidates.items():
    candidate = Agent(model_string, instructions="Answer with exactly one sentence.")
    r = await candidate.run(prompt)
    rows.append(
        {
            "model": label,
            "in": r.usage.input_tokens,
            "out": r.usage.output_tokens,
            "cost": r.usage.cost,
            "answer": r.output,
        }
    )
    # :16 pads the label to 16 characters; :>4 right-aligns each number in 4 characters
    print(
        f"{label:16} | in={r.usage.input_tokens:>4} out={r.usage.output_tokens:>4} cost={r.usage.cost}"
    )

priced = [row for row in rows if row["cost"] is not None]  # None = price unknown, not free
assert priced, "no model had a known price, so there is no cheapest"
# key= tells min() what to compare; lambda row: row["cost"] is a tiny inline function
cheapest = min(priced, key=lambda row: row["cost"])
print()
print("cheapest per answer:", cheapest["model"])

In [ ]:
assert len(rows) == len(candidates)
assert all(isinstance(row["answer"], str) and row["answer"].strip() for row in rows)
assert all(row["out"] > 0 for row in rows)
assert cheapest["cost"] is not None and all(cheapest["cost"] <= row["cost"] for row in priced)
print("all checks passed")